In [0]:
# ==================================================
# INGESTA BRONZE - TRACKING DE ENVIOS
# ORIGEN: Azure SQL Database
# DESTINO: Databricks Delta Lake
# ==================================================

# Tabla federada de Azure SQL Server
TABLA_ORIGEN = (
    "conexion_azure_sql_electrocasa_catalog"
    ".dbo.trackingenvios"
)

# Tabla destino en Bronze
TABLA_DESTINO = "proyecto_final.bronze.tracking_envios"

# Leer los datos desde Azure SQL
df_tracking = spark.table(TABLA_ORIGEN)

# Mostrar esquema original
print("ESQUEMA DEL ORIGEN:")
df_tracking.printSchema()

# Contar registros
total_origen = df_tracking.count()

print("TOTAL DE REGISTROS EN AZURE SQL:", total_origen)

# Mostrar muestra de datos
display(df_tracking.limit(10))

In [0]:
# ==================================================
# INGESTA BRONZE - TRACKING DE ENVIOS
# Guardar copia de Azure SQL en Delta Lake
# ==================================================

from pyspark.sql.functions import current_timestamp, lit

# Agregar campos de trazabilidad
df_bronze = (
    df_tracking
    .withColumn("fuente_origen", lit("Azure SQL Database"))
    .withColumn("fecha_ingesta", current_timestamp())
)

# Crear la tabla solo si todavía no existe
if not spark.catalog.tableExists(TABLA_DESTINO):

    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_DESTINO)
    )

    print("Tabla Bronze creada correctamente.")

else:
    print("La tabla Bronze ya existe. No se duplicaron registros.")

print("DESTINO:", TABLA_DESTINO)

In [0]:
%sql
-- ==================================================
-- VALIDACION BRONZE - TRACKING DE ENVIOS
-- ==================================================

SELECT
    fuente_origen,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT evento_id) AS eventos_distintos,
    MIN(fecha_ingesta) AS primera_ingesta,
    MAX(fecha_ingesta) AS ultima_ingesta
FROM proyecto_final.bronze.tracking_envios
GROUP BY fuente_origen;